# Low-Light Image Enhancement with an AI Assistant

## Step 1: Load an image and measure how dark it is

Before we enhance anything, the program must first *understand* the image.
In this step we measure brightness, contrast and noise, and turn the
brightness into a simple label (Normal, Dim, Dark, Very dark).
Later, Jarvis will use this label to decide which enhancement method to use.

In [ ]:
# Step 1.1 - Import the libraries we need
import cv2                       # OpenCV: reads and processes images
import numpy as np               # NumPy: fast maths on image arrays
import matplotlib.pyplot as plt  # Matplotlib: shows images and graphs

In [ ]:
# Step 1.2 - Load an image
# "sample"  -> uses a built-in test photo that we darken ourselves
# "upload"  -> lets you pick your own dark photo from your computer
IMAGE_SOURCE = "sample"

if IMAGE_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()                # a file picker opens
    file_name = list(uploaded.keys())[0]
    img = cv2.imread(file_name)              # OpenCV loads images in BGR order
else:
    from skimage import data
    bright = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)
    # Make it dark: raising pixel values (scaled 0-1) to a power above 1 darkens them
    img = ((bright / 255.0) ** 2.5 * 0.4 * 255).astype(np.uint8)

print("Image size (height, width, channels):", img.shape)

In [ ]:
# Step 1.3 - Functions that measure the image

def to_gray(image):
    # Brightness is easiest to measure on a single grey channel
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

def get_brightness(image):
    # Average pixel value scaled to 0-1 (0 = black, 1 = white)
    return to_gray(image).mean() / 255.0

def get_contrast(image):
    # Standard deviation of pixel values: how spread out the tones are
    return to_gray(image).std() / 255.0

def get_dark_ratio(image):
    # Fraction of pixels that are very dark (below 15% brightness)
    return (to_gray(image) < 0.15 * 255).mean()

def get_noise(image):
    # Noise estimate (Immerkaer's method): a filter that ignores smooth areas
    # and responds mostly to random grain
    gray = to_gray(image).astype(np.float32)
    h, w = gray.shape
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    filtered = cv2.filter2D(gray, -1, kernel)
    return np.sum(np.abs(filtered)) * np.sqrt(0.5 * np.pi) / (6.0 * (w - 2) * (h - 2))

def darkness_level(brightness):
    # Turn the brightness number into a label Jarvis can use later
    if brightness >= 0.40:
        return "Normal"
    if brightness >= 0.25:
        return "Dim"
    if brightness >= 0.12:
        return "Dark"
    return "Very dark"

def analyze_image(image):
    brightness = get_brightness(image)
    level = darkness_level(brightness)
    return {
        "Brightness (0-1)": round(brightness, 3),
        "Contrast (0-1)": round(get_contrast(image), 3),
        "Dark pixel ratio": round(get_dark_ratio(image), 3),
        "Noise level": round(get_noise(image), 2),
        "Darkness level": level,
        "Needs enhancement": level != "Normal",
    }

In [ ]:
# Step 1.4 - Analyse the image and show the result
stats = analyze_image(img)
for name, value in stats.items():
    print(f"{name:20s}: {value}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # matplotlib expects RGB
axes[0].set_title("Input image")
axes[0].axis("off")

axes[1].hist(to_gray(img).ravel(), bins=64, range=(0, 255), color="gray")
axes[1].set_title("Brightness histogram")
axes[1].set_xlabel("Pixel value (0 = black, 255 = white)")
axes[1].set_ylabel("Number of pixels")
plt.tight_layout()
plt.show()

## Step 2: Enhancement methods

Now that the program knows the image is dark, we need ways to brighten it.
We will implement four classic methods. Each one is a different idea:

| Method | Simple idea | Good for | Weakness |
|---|---|---|---|
| Gamma correction | Apply a curve that lifts dark pixels more than bright ones | Fast, simple brightening | Can wash out colours and amplify noise |
| Histogram equalization | Spread the pixel values so they use the full 0-255 range | Boosting overall contrast | Can look harsh and over-enhanced |
| CLAHE | Equalize small tiles separately, with a limit on contrast boost | Local detail without as much harshness | Needs the clip limit tuned |
| Retinex (SSR / MSR) | Estimate the lighting and remove it, keeping the scene's own detail | Natural-looking detail in shadows | Slower, and can create grey or noisy results |

In [ ]:
# Step 2.1 - Shrink big photos so the methods run quickly
# Phone photos can have over 10 million pixels. Limiting the longest side
# to 1000 pixels keeps everything fast and the pictures easy to compare.
def resize_if_large(image, max_side=1000):
    h, w = image.shape[:2]
    scale = max_side / max(h, w)
    if scale < 1:
        image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    return image

small = resize_if_large(img)
print("Working image size:", small.shape)

### Method 1: Gamma correction
Every pixel value (scaled to 0-1) is replaced by `value ** gamma`.
With gamma below 1 the curve rises steeply at the dark end, so dark pixels are lifted a lot
while bright pixels barely change. If no gamma is given, the function picks one that moves
the average brightness to about 0.45.

In [ ]:
# Step 2.2 - Gamma correction
def gamma_correction(image, gamma=None):
    if gamma is None:
        # Automatic gamma: aim for an average brightness of about 0.45
        mean = np.clip(get_brightness(image), 0.01, 0.99)
        gamma = np.log(0.45) / np.log(mean)
        gamma = np.clip(gamma, 0.25, 1.0)    # limit it so we do not over-brighten
    # Lookup table: the new value for each of the 256 possible pixel values
    table = np.array([(i / 255.0) ** gamma * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(image, table)

### Method 2: Histogram equalization
The histogram you saw in Step 1 is bunched up on the left. Histogram equalization stretches it
so the pixel values use the whole range from black to white. We apply it only to the
*brightness* channel (Y in the YCrCb colour space). If we equalized the blue, green and red
channels separately, the colours would shift.

In [ ]:
# Step 2.3 - Histogram equalization (on brightness only, so colours stay natural)
def histogram_equalization(image):
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
    return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)

### Method 3: CLAHE (Contrast Limited Adaptive Histogram Equalization)
Normal equalization treats the whole image the same. CLAHE splits the image into small tiles
(8 x 8 here) and equalizes each tile on its own, so shadows and highlights are handled
locally. The **clip limit** stops any tile from being boosted too much, which keeps noise
under control. We apply it to the L (lightness) channel of the LAB colour space.

In [ ]:
# Step 2.4 - CLAHE
def clahe_enhance(image, clip_limit=3.0, tile_size=8):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    engine = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    lab[:, :, 0] = engine.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

### Method 4: Retinex (SSR and MSR)
Retinex is based on a model of how we see: an image is the *scene itself* multiplied by the
*lighting*. In a dark photo the lighting is weak. We estimate the lighting by blurring the
image heavily, then remove it using logarithms: `log(image) - log(blurred image)`.

- **SSR** (Single Scale Retinex) uses one blur size (sigma).
- **MSR** (Multi Scale Retinex) averages several blur sizes: a small one for fine detail,
  a large one for overall lighting.

In [ ]:
# Step 2.5 - Retinex
def retinex(image, sigmas):
    img_f = image.astype(np.float32) + 1.0        # +1 avoids log(0) on black pixels
    result = np.zeros_like(img_f)
    for c in range(3):                            # blue, green, red channels
        channel = img_f[:, :, c]
        total = 0
        for sigma in sigmas:
            blurred = cv2.GaussianBlur(channel, (0, 0), sigma)   # estimate of the lighting
            total = total + (np.log10(channel) - np.log10(blurred))
        result[:, :, c] = total / len(sigmas)
    # Stretch to 0-255 (ignore the most extreme 1% at each end)
    low, high = np.percentile(result, [1, 99])
    result = np.clip((result - low) / max(high - low, 1e-6), 0, 1)
    return (result * 255).astype(np.uint8)

def retinex_ssr(image, sigma=80):
    return retinex(image, (sigma,))

def retinex_msr(image, sigmas=(15, 80, 250)):
    return retinex(image, sigmas)

In [ ]:
# Step 2.6 - Run all four methods and compare them side by side
results = {
    "Original": small,
    "Gamma correction": gamma_correction(small),
    "Histogram equalization": histogram_equalization(small),
    "CLAHE": clahe_enhance(small),
    "Retinex SSR": retinex_ssr(small),
    "Retinex MSR": retinex_msr(small),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, (name, image) in zip(axes.ravel(), results.items()):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{name}\nbrightness = {get_brightness(image):.2f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 3: Remove blocky artifacts and noise *before* enhancing

**What went wrong in Step 2?** Look at the results: you can see stair-step edges and
blocky coloured patches. A very dark photo stores very little information: only a few
brightness levels, plus compression blocks from the JPEG file. Enhancement multiplies tiny
differences by 10 to 50 times, so these hidden flaws become visible.

**The fix:** smooth the dark image first, then enhance it. Two details matter:

1. **Use decimal (float) numbers, not whole numbers.** If we smooth and then round back to
   whole numbers 0-255, the stair-steps come back. So we keep decimals until the very end.
2. **Smooth colour much more than brightness.** The ugly blocks are mostly *colour* blocks.
   We split the image into brightness (Y) and colour (Cr, Cb), blur brightness lightly to keep
   detail, and blur colour strongly.

In [ ]:
# Step 3.1 - Smooth a dark image (returns decimal numbers, NOT rounded)
def smooth_dark_image(image, strength=1.0):
    # Blur sizes grow with the image size so the effect looks the same on any photo
    scale = max(image.shape[:2]) / 600.0
    luma_sigma = 1.5 * strength * scale       # light blur for brightness (keeps detail)
    chroma_sigma = 8.0 * strength * scale     # strong blur for colour (hides colour blocks)

    # Convert to brightness + colour using decimal numbers
    ycrcb = cv2.cvtColor(image.astype(np.float32) / 255.0, cv2.COLOR_BGR2YCrCb)
    ycrcb[:, :, 0] = cv2.GaussianBlur(ycrcb[:, :, 0], (0, 0), luma_sigma)
    ycrcb[:, :, 1] = cv2.GaussianBlur(ycrcb[:, :, 1], (0, 0), chroma_sigma)
    ycrcb[:, :, 2] = cv2.GaussianBlur(ycrcb[:, :, 2], (0, 0), chroma_sigma)

    smooth = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR) * 255.0
    return np.clip(smooth, 0, 255).astype(np.float32)

# Gamma correction that works on decimal pixels (Step 2.2 used a whole-number lookup table)
def gamma_correction_float(image_f, gamma=None):
    if gamma is None:
        mean = np.clip(get_brightness(image_f), 0.01, 0.99)
        gamma = np.clip(np.log(0.45) / np.log(mean), 0.25, 1.0)
    out = np.power(np.clip(image_f, 0, 255) / 255.0, gamma) * 255.0
    return np.clip(out, 0, 255).astype(np.uint8)

In [ ]:
# Step 3.2 - Compare: enhance directly vs smooth first, then enhance
# strength = 1.0 is the default. If you still see blocks, try 2.0. If it is too blurry, try 0.5.
smooth_1 = smooth_dark_image(small, strength=1.0)
smooth_2 = smooth_dark_image(small, strength=2.0)

results_3 = {
    "Gamma, no smoothing": gamma_correction_float(small.astype(np.float32)),
    "Gamma, smoothed (strength 1)": gamma_correction_float(smooth_1),
    "Gamma, smoothed (strength 2)": gamma_correction_float(smooth_2),
    "Retinex MSR, no smoothing": retinex_msr(small.astype(np.float32)),
    "Retinex MSR, smoothed (strength 1)": retinex_msr(smooth_1),
    "Retinex MSR, smoothed (strength 2)": retinex_msr(smooth_2),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, (name, image) in zip(axes.ravel(), results_3.items()):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 4: Measure how good each method is

Looking at pictures is not enough for a report: we need numbers. There are two kinds of metrics.

**With a reference photo** (the same scene taken in good light):
- **PSNR** (Peak Signal-to-Noise Ratio, in dB): based on the average pixel error compared with the
  reference. Higher is better.
- **SSIM** (Structural Similarity, 0 to 1): compares structure, contrast and brightness patterns.
  Closer to 1 is better.

**Without a reference** (works on your own photo):
- **Brightness**: average brightness (0-1).
- **Contrast**: spread of the tones.
- **Entropy**: amount of information in the image. Higher usually means more visible detail,
  but noise also raises it, so read it together with the picture.

Dark/bright pairs come from the **LOL dataset**, the standard benchmark for low-light enhancement.
Note: PSNR and SSIM do not always agree with what looks good to the eye, so we use both numbers
and the pictures.

In [ ]:
# Step 4.1 - Metric functions
import pandas as pd
from skimage.metrics import structural_similarity

try:
    from IPython.display import display
except ImportError:
    display = print

def get_psnr(reference, test):
    # Mean squared error between the two images, converted to decibels
    mse = np.mean((reference.astype(np.float64) - test.astype(np.float64)) ** 2)
    if mse == 0:
        return float("inf")
    return 10 * np.log10(255.0 ** 2 / mse)

def get_ssim(reference, test):
    return structural_similarity(reference, test, channel_axis=2, data_range=255)

def get_entropy(image):
    # Shannon entropy of the brightness histogram
    hist = cv2.calcHist([to_gray(image)], [0], None, [256], [0, 256]).ravel()
    p = hist / hist.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

# All the methods we have built, in one place. Each takes a dark image and returns a brighter one.
methods = {
    "Gamma": gamma_correction,
    "Histogram equalization": histogram_equalization,
    "CLAHE": clahe_enhance,
    "Retinex SSR": retinex_ssr,
    "Retinex MSR": retinex_msr,
    "Smooth + Gamma": lambda im: gamma_correction_float(smooth_dark_image(im)),
    "Smooth + Retinex MSR": lambda im: retinex_msr(smooth_dark_image(im)),
}

In [ ]:
# Step 4.2 - Numbers for YOUR photo (no reference needed)
rows = [{"Method": "Original (no enhancement)",
         "Brightness": get_brightness(small), "Contrast": get_contrast(small),
         "Entropy": get_entropy(small), "Noise": get_noise(small)}]
for name, fn in methods.items():
    out = fn(small)
    rows.append({"Method": name,
                 "Brightness": get_brightness(out), "Contrast": get_contrast(out),
                 "Entropy": get_entropy(out), "Noise": get_noise(out)})

photo_table = pd.DataFrame(rows).round(3)
display(photo_table)
photo_table.to_csv("my_photo_metrics.csv", index=False)

### Loading the LOL dataset
The next cell downloads the LOL dataset from Hugging Face (about 350 MB, one time), finds the
folders of dark ("low") and well-lit ("high") photos, and pairs them by file name.
We use up to 15 pairs, which is the size of the standard LOL test set.
If the download fails, the notebook says so clearly and uses demo pairs instead, so you will
know the numbers do not come from LOL.

In [ ]:
# Step 4.3 - Get pairs of (dark photo, well-lit photo)
import os, zipfile

def find_pairs_in_folder(root, max_pairs=15):
    # Look for sibling folders called "low" and "high" and pair the images by file name
    candidates = []
    for folder, subdirs, files in os.walk(root):
        names = {d.lower(): d for d in subdirs}
        if "low" in names and "high" in names:
            candidates.append((folder, names["low"], names["high"]))
    if not candidates:
        return [], None
    # Prefer the small test split (usually called eval15 or test) if it exists
    candidates.sort(key=lambda c: 0 if ("eval" in c[0].lower() or "test" in c[0].lower()) else 1)
    folder, low_name, high_name = candidates[0]
    low_dir, high_dir = os.path.join(folder, low_name), os.path.join(folder, high_name)
    pairs = []
    for file_name in sorted(os.listdir(low_dir)):
        high_path = os.path.join(high_dir, file_name)
        if not os.path.exists(high_path):
            continue
        low = cv2.imread(os.path.join(low_dir, file_name))
        high = cv2.imread(high_path)
        if low is not None and high is not None and low.shape == high.shape:
            pairs.append((file_name, low, high))
        if len(pairs) >= max_pairs:
            break
    return pairs, folder

def load_lol_pairs(max_pairs=15):
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        # Install the downloader if this Colab session does not have it
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
        from huggingface_hub import hf_hub_download
    zip_path = hf_hub_download(repo_id="geekyrakshit/LoL-Dataset",
                               filename="lol_dataset.zip", repo_type="dataset")
    if not os.path.isdir("lol_data"):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall("lol_data")
    return find_pairs_in_folder("lol_data", max_pairs)

def make_demo_pairs():
    # Backup plan: darken normal photos ourselves so we still have a reference
    from skimage import data
    rng = np.random.default_rng(0)
    sources = {"astronaut": data.astronaut(), "coffee": data.coffee(),
               "chelsea": data.chelsea(), "rocket": data.rocket()}
    demo = []
    for name, rgb in sources.items():
        bright = cv2.resize(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), (600, 400))
        for gamma, scale in [(2.0, 0.5), (2.6, 0.35)]:
            dark = (bright / 255.0) ** gamma * scale
            dark = np.clip(dark + rng.normal(0, 0.01, dark.shape), 0, 1)
            demo.append((f"{name}_gamma{gamma}", (dark * 255).astype(np.uint8), bright))
    return demo

try:
    pairs, found_in = load_lol_pairs()
    if not pairs:
        raise RuntimeError("no matching low/high folders were found inside the zip")
    DATASET_NAME = "LOL dataset"
    print(f"Loaded {len(pairs)} dark/well-lit pairs from: {found_in}")
except Exception as error:
    print("Could not load the LOL dataset:", error)
    print("Using DEMO pairs instead (normal photos that we darkened). Do not report these as LOL results.")
    pairs = make_demo_pairs()
    DATASET_NAME = "demo images (not LOL)"
    print(f"Created {len(pairs)} demo pairs")

In [ ]:
# Step 4.4 - Score every method on every pair
all_methods = {"No enhancement": (lambda im: im), **methods}
scores = {name: {"psnr": [], "ssim": [], "brightness": []} for name in all_methods}

for file_name, low, high in pairs:
    for name, fn in all_methods.items():
        out = fn(low)
        scores[name]["psnr"].append(get_psnr(high, out))
        scores[name]["ssim"].append(get_ssim(high, out))
        scores[name]["brightness"].append(get_brightness(out))

summary = pd.DataFrame([{
    "Method": name,
    "PSNR (dB)": np.mean(s["psnr"]),
    "SSIM": np.mean(s["ssim"]),
    "Brightness": np.mean(s["brightness"]),
} for name, s in scores.items()]).round(3)

print(f"Average over {len(pairs)} image pairs - {DATASET_NAME}")
display(summary.sort_values("PSNR (dB)", ascending=False).reset_index(drop=True))
summary.to_csv("results_table.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].barh(summary["Method"], summary["PSNR (dB)"], color="gray")
axes[0].set_title("Average PSNR (higher is better)")
axes[0].set_xlabel("dB")
axes[1].barh(summary["Method"], summary["SSIM"], color="gray")
axes[1].set_title("Average SSIM (higher is better)")
axes[1].set_xlabel("SSIM (0 to 1)")
plt.tight_layout()
plt.savefig("psnr_ssim_chart.png", dpi=150)
plt.show()

In [ ]:
# Step 4.5 - One example pair: dark input, every method, and the well-lit reference
file_name, low, high = pairs[0]
panels = {"Dark input": low}
for name, fn in methods.items():
    panels[name] = fn(low)
panels["Well-lit reference"] = high

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, (name, image) in zip(axes.ravel(), panels.items()):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.savefig("example_comparison.png", dpi=150)
plt.show()